# Interactive Exploration of the OctreeHybridMeshGeneratorModeler

This notebook demonstrates every stage of the `KratosMultiphysics.OctreeHybridMeshGeneratorModeler`
pipeline using simple Python-constructed surface meshes and PyVista for 3-D visualisation.

**Pipeline overview** — `SetupModelPart` executes four sequential stages:

| Stage | JSON key | What it does |
|-------|----------|--------------|
| 1 | `refine_operations_list` | Builds the octree from the input surface (`OctreeHybridRefineInterfaceCells`) and applies optional additional refinement passes before 2:1 balancing and mesh extraction. |
| 2 | `coloring_settings_list` | Classifies cells as inside (1) or outside (0) the input surface. |
| 3 | `entities_generator_list` | Emits hexahedral/tetrahedral elements, surface conditions, and/or hanging-node constraints. |
| 4 | `model_part_operations` | Post-processing passes (e.g. mesh-quality statistics). |

**Components covered:**
- `OctreeHybridRefineInterfaceCells` — builds the adaptive octree from a triangulated surface (always the first entry of `refine_operations_list`).
- `OctreeHybridRefineUniform` — uniform depth refinement of all octree cells.
- `OctreeHybridRefineInterfaceCells` — interface-only refinement driven by a surface model part (chainable for multiple surfaces).
- `OctreeHybridClassifyCellsInsideOutside` — inside/outside colouring via ray-cast + signed distance.
- `GenerateHybridOctreeHexahedraElementsWithCellColor` — one `Element3D8N` per cell matching a colour value.
- `GenerateHybridOctreeQuadrilateralConditionsWithFaceColor` — `SurfaceCondition3D4N` on the outer boundary.
- `GenerateHybridOctreeTetrahedraElementsWithCellColor` — six `Element3D4N` per hex cell (BCC Freudenthal decomposition, §3.3.3).
- `GenerateHybridOctreeTriangularConditionsWithFaceColor` — `SurfaceCondition3D3N` on the outer boundary (2 per boundary quad).
- `GenerateHybridOctreeHexahedraElementsWithCellColor` — also generates `LinearMasterSlaveConstraint` for primal-mesh 2:1 transitions when `"variables"` is non-empty.
- `OctreeHybridReportMeshQuality` — scaled-Jacobian statistics logged to the Kratos output stream.

### Enabling Interactive 3-D Rendering

By default this notebook renders **static** PNG images.  To enable interactive rotate/pan/zoom:
1. Install the required packages:
   ```bash
   pip install trame trame-vtk trame-vuetify ipywidgets nest-asyncio
   ```
2. Un-comment and run the interactive backend lines in the imports cell below.

In [ ]:
import sys
import os

# Resolve the Kratos repo root relative to this notebook's location:
# notebooks/ -> python_scripts/ -> kratos/ -> <repo root>
# NOTE: Only needed for self-compiled Kratos; skip for system-wide installations.
notebook_dir = os.path.dirname(os.path.abspath("__file__"))
kratos_root = os.path.abspath(os.path.join(notebook_dir, "..", "..", ".."))
kratos_build_path = os.path.join(kratos_root, "bin", "Release")

if kratos_build_path not in sys.path:
    sys.path.insert(0, kratos_build_path)

print(f"Kratos build path: {kratos_build_path}")

In [ ]:
import collections
import tempfile

import numpy as np
import pyvista as pv
import KratosMultiphysics as KM
import KratosMultiphysics.pyvista_utilities as pv_utils

pv.global_theme.color = "white"

# ==========================================================================
# INTERACTIVE 3-D VISUALISATION IN JUPYTER NOTEBOOKS:
# 1. pip install trame trame-vtk trame-vuetify ipywidgets nest-asyncio
# 2. Un-comment the lines below:
#    import nest_asyncio
#    nest_asyncio.apply()
#    pv.set_jupyter_backend("client")
# ==========================================================================

print("Modules imported successfully!")

### Step 1: Build Input Surface Geometry

The modeler requires a **closed triangulated surface** stored as `Triangle3D3` geometries inside
a Kratos `ModelPart`.  Two helper functions are defined here:

- **`build_closed_box_surface`** — a simple axis-aligned cube `[lo, hi]³`, triangulated into
  12 triangles (2 per face).  Two extra bounding-box pin nodes (at the unit-cube corners) are
  added as required by the octree engine.
- **`build_transition_surface`** — a small inclined quad patch near one corner of the unit cube.
  Being far smaller than the domain, it forces multiple levels of 2:1 refinement in the adaptive
  octree — useful for testing the primal mesh and hanging-node constraints.

> **Note:** The two bounding-box pin nodes (`[0,0,0]` and `[1,1,1]`) tell the octree engine the
> extent of the mesh domain.  They do **not** need to be part of any triangle.

In [ ]:
def build_closed_box_surface(model, lo=0.3, hi=0.7, name="Surface"):
    """Closed triangulated cube [lo, hi]^3 with two bounding-box pin nodes."""
    mp = model.CreateModelPart(name)
    mp.ProcessInfo[KM.DOMAIN_SIZE] = 3

    corners = [
        (lo, lo, lo), (hi, lo, lo), (hi, hi, lo), (lo, hi, lo),
        (lo, lo, hi), (hi, lo, hi), (hi, hi, hi), (lo, hi, hi),
    ]
    for i, (x, y, z) in enumerate(corners, start=1):
        mp.CreateNewNode(i, x, y, z)
    mp.CreateNewNode(9,  0.0, 0.0, 0.0)   # bbox pin (min)
    mp.CreateNewNode(10, 1.0, 1.0, 1.0)   # bbox pin (max)

    # 12 triangles — 2 per cube face (0-indexed corner offsets)
    faces = [
        (0,1,2),(0,2,3),  # -Z face
        (4,6,5),(4,7,6),  # +Z face
        (0,5,1),(0,4,5),  # -Y face
        (3,2,6),(3,6,7),  # +Y face
        (0,3,7),(0,7,4),  # -X face
        (1,5,6),(1,6,2),  # +X face
    ]
    for gid, (a, b, c) in enumerate(faces, start=1):
        mp.CreateNewGeometry("Triangle3D3", gid, [a+1, b+1, c+1])
    return mp


def build_transition_surface(model, name="Surface"):
    """Small inclined patch that forces 2:1 transitions in the adaptive octree."""
    mp = model.CreateModelPart(name)
    mp.ProcessInfo[KM.DOMAIN_SIZE] = 3

    pts = [
        (0.0, 0.0, 0.0), (1.0, 1.0, 1.0),   # bbox pins
        (0.15, 0.15, 0.30), (0.45, 0.15, 0.30),
        (0.45, 0.45, 0.36), (0.15, 0.45, 0.36),
    ]
    for i, (x, y, z) in enumerate(pts, start=1):
        mp.CreateNewNode(i, x, y, z)
    mp.CreateNewGeometry("Triangle3D3", 1, [3, 4, 5])
    mp.CreateNewGeometry("Triangle3D3", 2, [3, 5, 6])
    return mp


# Build the primary working surface for Steps 2–5
model = KM.Model()
build_closed_box_surface(model, lo=0.3, hi=0.7, name="Surface")
print(f"Surface model part: {model.GetModelPart('Surface').NumberOfNodes()} nodes, "
      f"{model.GetModelPart('Surface').NumberOfGeometries()} triangles")

### Step 2: Dual Hex Mesh — Minimal Pipeline

The *dual* mesh is the default topology: the octree is 2:1-balanced, and the dual of each
interior primal vertex becomes one conforming hexahedron.  Transition templates stitch cells
at refinement boundaries so the mesh has **no hanging nodes**.

The minimal pipeline needs:
1. **`OctreeHybridRefineInterfaceCells`** (first entry of `refine_operations_list`) — builds the octree from the surface.  Optional further entries can refine it before extraction.
2. **`OctreeHybridClassifyCellsInsideOutside`** — marks interior cells with colour `1`.
3. **`GenerateHybridOctreeHexahedraElementsWithCellColor`** — emits `Element3D8N` for every `color=1` cell.

The example below passes `OctreeHybridRefineInterfaceCells` at depth 1 and then chains
`OctreeHybridRefineUniform` to depth 4 — keeping the generation step separate from the
refinement step.

In [ ]:
settings_dual = KM.Parameters("""
{
    "input_model_part_name"  : "Surface",
    "refine_operations_list" : [
        { "type": "OctreeHybridRefineInterfaceCells", "adaptive": false, "mesh_type": "dual" },
        { "type": "OctreeHybridRefineUniform", "refinement_depth": 4 }
    ],
    "coloring_settings_list"  : [{ "type": "OctreeHybridClassifyCellsInsideOutside" }],
    "entities_generator_list" : [{
        "type"               : "GenerateHybridOctreeHexahedraElementsWithCellColor",
        "model_part_name"    : "Volume",
        "color"              : 1,
        "tag_refinement_level": true
    }],
    "model_part_operations" : []
}
""")

mod = KM.OctreeHybridMeshGeneratorModeler(model, settings_dual)
mod.SetupModelPart()

mp_vol = model.GetModelPart("Volume")
print(f"Dual hex mesh (uniform, depth=4):")
print(f"  Nodes    : {mp_vol.NumberOfNodes()}")
print(f"  Elements : {mp_vol.NumberOfElements()}")

# Quick quality check: no inverted elements
SJ_ADJ = [(1,3,4),(2,0,5),(3,1,6),(0,2,7),(7,5,0),(4,6,1),(5,7,2),(6,4,3)]

def _sub(p, q): return (p[0]-q[0], p[1]-q[1], p[2]-q[2])
def _norm(v):   return (v[0]**2 + v[1]**2 + v[2]**2)**0.5
def _dot(a, b): return a[0]*b[0] + a[1]*b[1] + a[2]*b[2]
def _cross(a, b): return (a[1]*b[2]-a[2]*b[1], a[2]*b[0]-a[0]*b[2], a[0]*b[1]-a[1]*b[0])

def min_scaled_jacobian(element):
    geom = element.GetGeometry()
    coords = [(geom[i].X, geom[i].Y, geom[i].Z) for i in range(8)]
    worst = 1e30
    for o, (x, y, z) in enumerate(SJ_ADJ):
        e1 = _sub(coords[x], coords[o])
        e2 = _sub(coords[y], coords[o])
        e3 = _sub(coords[z], coords[o])
        n1, n2, n3 = _norm(e1), _norm(e2), _norm(e3)
        if min(n1, n2, n3) < 1e-14:
            return -1.0
        worst = min(worst, _dot(e1, _cross(e2, e3)) / (n1 * n2 * n3))
    return worst

n_inverted = sum(1 for el in mp_vol.Elements if min_scaled_jacobian(el) <= 0)
all_sj = [min_scaled_jacobian(el) for el in mp_vol.Elements]
print(f"  Inverted elements  : {n_inverted}")
print(f"  Min scaled Jacobian: {min(all_sj):.6f}")
print(f"  Mean scaled Jacobian: {sum(all_sj)/len(all_sj):.6f}")

In [ ]:
# Visualise the dual hex mesh.
# Since the generated nodes carry no simulation variables, we add a synthetic
# point field (distance from the domain centre) for colouring.
grid = pv_utils.ModelPartToPyVista(mp_vol)

cx, cy, cz = 0.5, 0.5, 0.5
grid.point_data["dist_to_centre"] = np.array(
    [((n.X - cx)**2 + (n.Y - cy)**2 + (n.Z - cz)**2)**0.5 for n in mp_vol.Nodes]
)

grid.plot(
    scalars="dist_to_centre",
    show_edges=True,
    cmap="viridis",
    cpos="iso",
    text="Dual Hex Mesh — carved box [0.3, 0.7]³",
)

### Step 3: Refinement Operations — `refine_operations_list`

The **`refine_operations_list`** stage runs **after** the initial octree is built from the
surface mesh and **before** the mandatory 2:1 balancing (`StrongConstrain2To1`) and mesh
extraction.  All refinement passes accumulate in a single octree and are balanced in one
call, keeping the pipeline efficient.

Two ready-to-use refinement operations are provided:

| Operation | JSON `type` key | Purpose |
|---|---|---|
| `OctreeHybridRefineUniform` | `"OctreeHybridRefineUniform"` | Subdivides **all** leaf cells to a prescribed depth, producing a globally uniform octree. |
| `OctreeHybridRefineInterfaceCells` | `"OctreeHybridRefineInterfaceCells"` | Subdivides only cells that contain surface-triangle vertices, concentrating resolution near the specified geometry. |

Both accept a `"refinement_depth"` key.  `OctreeHybridRefineInterfaceCells` also accepts
`"input_model_part_name"` — when non-empty, the triangle soup is re-extracted from that model
part, allowing **multiple chained entries targeting different surfaces**.

```json
"refine_operations_list": [
  { "type": "OctreeHybridRefineInterfaceCells",
    "input_model_part_name": "InnerWall", "refinement_depth": 6 },
  { "type": "OctreeHybridRefineInterfaceCells",
    "input_model_part_name": "OuterWall", "refinement_depth": 4 }
]
```


In [ ]:
# --- Uniform refinement via OctreeHybridRefineUniform ---
# Generate the octree at depth 1 (via OctreeHybridRefineInterfaceCells), then refine
# to depth 4 via OctreeHybridRefineUniform.
# The element count must match a direct depth-4 build (same octree topology).

model_uniform = KM.Model()
build_closed_box_surface(model_uniform, name="Surface")

mod_uniform = KM.OctreeHybridMeshGeneratorModeler(model_uniform, KM.Parameters("""
{
    "input_model_part_name"  : "Surface",
    "refine_operations_list" : [
        { "type": "OctreeHybridRefineInterfaceCells", "refinement_depth": 1, "adaptive": false },
        { "type": "OctreeHybridRefineUniform", "refinement_depth": 4 }
    ],
    "coloring_settings_list"  : [{ "type": "OctreeHybridClassifyCellsInsideOutside" }],
    "entities_generator_list" : [{
        "type"                : "GenerateHybridOctreeHexahedraElementsWithCellColor",
        "model_part_name"     : "Volume",
        "color"               : 1,
        "tag_refinement_level": true
    }],
    "model_part_operations" : []
}
"""))
mod_uniform.SetupModelPart()

mp_uniform = model_uniform.GetModelPart("Volume")
print(f"Uniform refinement (depth 1 → 4 via OctreeHybridRefineUniform):")
print(f"  Nodes    : {mp_uniform.NumberOfNodes()}")
print(f"  Elements : {mp_uniform.NumberOfElements()}")

# Reference: direct depth-4 build
model_ref = KM.Model()
build_closed_box_surface(model_ref, name="Surface")
mod_ref = KM.OctreeHybridMeshGeneratorModeler(model_ref, KM.Parameters("""
{
    "input_model_part_name" : "Surface",
    "refine_operations_list": [
        { "type": "OctreeHybridRefineInterfaceCells", "refinement_depth": 4, "adaptive": false }
    ],
    "coloring_settings_list" : [{ "type": "OctreeHybridClassifyCellsInsideOutside" }],
    "entities_generator_list": [{
        "type": "GenerateHybridOctreeHexahedraElementsWithCellColor",
        "model_part_name": "Volume", "color": 1
    }],
    "model_part_operations": []
}
"""))
mod_ref.SetupModelPart()
mp_ref = model_ref.GetModelPart("Volume")
print(f"Direct depth-4 reference  :")
print(f"  Nodes    : {mp_ref.NumberOfNodes()}")
print(f"  Elements : {mp_ref.NumberOfElements()}")
match = mp_uniform.NumberOfElements() == mp_ref.NumberOfElements()
print(f"\nElement counts match: {match}  (expected: True)")

In [ ]:
# --- Interface-cell refinement via OctreeHybridRefineInterfaceCells ---
# Only cells containing surface-triangle vertices are refined to depth 4;
# interior and far-field cells remain at the coarser initial depth (2).
# This produces a graded mesh without the overhead of a fully uniform octree.

model_iface = KM.Model()
build_closed_box_surface(model_iface, name="Surface")

mod_iface = KM.OctreeHybridMeshGeneratorModeler(model_iface, KM.Parameters("""
{
    "input_model_part_name"  : "Surface",
    "refine_operations_list" : [
        { "type": "OctreeHybridRefineInterfaceCells", "refinement_depth": 2, "adaptive": false },
        {
            "type"                  : "OctreeHybridRefineInterfaceCells",
            "input_model_part_name" : "Surface",
            "refinement_depth"      : 4
        }
    ],
    "coloring_settings_list"  : [{ "type": "OctreeHybridClassifyCellsInsideOutside" }],
    "entities_generator_list" : [{
        "type"                : "GenerateHybridOctreeHexahedraElementsWithCellColor",
        "model_part_name"     : "Volume",
        "color"               : 1,
        "tag_refinement_level": true
    }],
    "model_part_operations" : []
}
"""))
mod_iface.SetupModelPart()

mp_iface = model_iface.GetModelPart("Volume")
print(f"Interface-cell refinement (base depth 2, interface depth 4):")
print(f"  Nodes    : {mp_iface.NumberOfNodes()}")
print(f"  Elements : {mp_iface.NumberOfElements()}")

levels_iface = [el.GetValue(KM.REFINEMENT_LEVEL) for el in mp_iface.Elements]
print(f"  REFINEMENT_LEVEL distribution: "
      f"{dict(sorted(collections.Counter(levels_iface).items()))}")
print(f"  (coarse interior cells at lower levels, fine interface cells at level 4)")

# Visualise
grid_iface = pv_utils.ModelPartToPyVista(mp_iface)
grid_iface.cell_data["REFINEMENT_LEVEL"] = np.array(
    [el.GetValue(KM.REFINEMENT_LEVEL) for el in mp_iface.Elements]
)
grid_iface.plot(
    scalars="REFINEMENT_LEVEL",
    show_edges=True,
    cmap="tab10",
    cpos="iso",
    text="Interface-Cell Refinement — fine near surface, coarse interior",
)

In [ ]:
# --- Multiple chained OctreeHybridRefineInterfaceCells entries ---
# Two separate box surfaces at different positions; each gets its own
# refinement depth.  The resulting octree concentrates cells around
# both surfaces simultaneously before a single StrongConstrain2To1 call.

model_multi = KM.Model()
build_closed_box_surface(model_multi, lo=0.15, hi=0.45, name="BoxA")
build_closed_box_surface(model_multi, lo=0.55, hi=0.85, name="BoxB")

# BoxA drives the octree bounding box; BoxB is passed via input_model_part_name.
mod_multi = KM.OctreeHybridMeshGeneratorModeler(model_multi, KM.Parameters("""
{
    "input_model_part_name"  : "BoxA",
    "refine_operations_list" : [
        { "type": "OctreeHybridRefineInterfaceCells", "refinement_depth": 1, "adaptive": false },
        { "type": "OctreeHybridRefineInterfaceCells",
          "input_model_part_name": "BoxA", "refinement_depth": 4 },
        { "type": "OctreeHybridRefineInterfaceCells",
          "input_model_part_name": "BoxB", "refinement_depth": 3 }
    ],
    "coloring_settings_list"  : [{ "type": "OctreeHybridClassifyCellsInsideOutside" }],
    "entities_generator_list" : [{
        "type"                : "GenerateHybridOctreeHexahedraElementsWithCellColor",
        "model_part_name"     : "Volume",
        "color"               : 1,
        "tag_refinement_level": true
    }],
    "model_part_operations" : []
}
"""))
mod_multi.SetupModelPart()

mp_multi = model_multi.GetModelPart("Volume")
print(f"Multi-surface chained refinement:")
print(f"  Nodes    : {mp_multi.NumberOfNodes()}")
print(f"  Elements : {mp_multi.NumberOfElements()}")

grid_multi = pv_utils.ModelPartToPyVista(mp_multi)
grid_multi.cell_data["REFINEMENT_LEVEL"] = np.array(
    [el.GetValue(KM.REFINEMENT_LEVEL) for el in mp_multi.Elements]
)
grid_multi.plot(
    scalars="REFINEMENT_LEVEL",
    show_edges=True,
    cmap="tab10",
    cpos="iso",
    text="Multi-Surface Refinement — BoxA @ depth 4, BoxB @ depth 3",
)

In [ ]:
# --- element_size parameter: world-space units instead of depth ---
# When element_size > 0 it overrides refinement_depth by computing the equivalent
# octree depth from the bounding box.  Useful when the domain size is known but
# the correct depth level is not obvious.

model_esize = KM.Model()
build_closed_box_surface(model_esize, name="Surface")

# The box is [0, 1]^3.  element_size=0.125 → depth 3 (8 cells per axis),
# element_size=0.0625 → depth 4 (16 cells per axis).
mod_esize = KM.OctreeHybridMeshGeneratorModeler(model_esize, KM.Parameters("""
{
    "input_model_part_name"  : "Surface",
    "refine_operations_list" : [
        { "type": "OctreeHybridRefineInterfaceCells", "adaptive": false },
        { "type": "OctreeHybridRefineUniform", "element_size": 0.0625 }
    ],
    "coloring_settings_list"  : [{ "type": "OctreeHybridClassifyCellsInsideOutside" }],
    "entities_generator_list" : [{
        "type"                : "GenerateHybridOctreeHexahedraElementsWithCellColor",
        "model_part_name"     : "Volume",
        "color"               : 1,
        "tag_refinement_level": true
    }],
    "model_part_operations" : []
}
"""))
mod_esize.SetupModelPart()

mp_esize = model_esize.GetModelPart("Volume")
print(f"element_size=0.0625 (≈ depth 4 for unit box):")
print(f"  Nodes    : {mp_esize.NumberOfNodes()}")
print(f"  Elements : {mp_esize.NumberOfElements()}")

# Compare with explicit refinement_depth=4
model_d4 = KM.Model()
build_closed_box_surface(model_d4, name="Surface")
mod_d4 = KM.OctreeHybridMeshGeneratorModeler(model_d4, KM.Parameters("""
{
    "input_model_part_name"  : "Surface",
    "refine_operations_list" : [
        { "type": "OctreeHybridRefineInterfaceCells", "adaptive": false },
        { "type": "OctreeHybridRefineUniform", "refinement_depth": 4 }
    ],
    "coloring_settings_list"  : [{ "type": "OctreeHybridClassifyCellsInsideOutside" }],
    "entities_generator_list" : [{ "type": "GenerateHybridOctreeHexahedraElementsWithCellColor",
                                    "model_part_name": "Volume", "color": 1 }],
    "model_part_operations" : []
}
"""))
mod_d4.SetupModelPart()
mp_d4 = model_d4.GetModelPart("Volume")
print(f"refinement_depth=4 reference:")
print(f"  Nodes    : {mp_d4.NumberOfNodes()}")
print(f"  Elements : {mp_d4.NumberOfElements()}")
match = mp_esize.NumberOfElements() == mp_d4.NumberOfElements()
print(f"\nElement counts match: {match}  (expected: True)")

### Step 3: Inside/Outside Colouring — What `OctreeHybridClassifyCellsInsideOutside` Does

`OctreeHybridClassifyCellsInsideOutside` classifies every octree cell by a ray-cast parity test combined
with a closest-triangle signed distance.  Cells with colour `1` are *inside* the surface;
colour `0` cells are *outside*.

The cell below runs the pipeline **with** and **without** the colouring stage to show how many
cells are removed when the inside-only filter is applied.

In [ ]:
# --- Unfiltered: no colouring, all cells emitted (color=1 default means all pass) ---
model_all = KM.Model()
build_closed_box_surface(model_all, name="Surface")
mod_all = KM.OctreeHybridMeshGeneratorModeler(model_all, KM.Parameters("""
{
    "input_model_part_name" : "Surface",
    "refine_operations_list": [
        {"type": "OctreeHybridRefineInterfaceCells", "adaptive": false},
        {"type": "OctreeHybridRefineUniform", "refinement_depth": 4}
    ],
    "coloring_settings_list" : [],
    "entities_generator_list": [{"type": "GenerateHybridOctreeHexahedraElementsWithCellColor",
                                  "model_part_name": "All", "color": 1}],
    "model_part_operations"  : []
}
"""))
mod_all.SetupModelPart()
n_all = model_all.GetModelPart("All").NumberOfElements()

# --- Carved: ClassifyCellsInsideOutside removes outside cells ---
model_carved = KM.Model()
build_closed_box_surface(model_carved, name="Surface")
mod_carved = KM.OctreeHybridMeshGeneratorModeler(model_carved, KM.Parameters("""
{
    "input_model_part_name" : "Surface",
    "refine_operations_list": [
        {"type": "OctreeHybridRefineInterfaceCells", "adaptive": false},
        {"type": "OctreeHybridRefineUniform", "refinement_depth": 4}
    ],
    "coloring_settings_list" : [{"type": "OctreeHybridClassifyCellsInsideOutside"}],
    "entities_generator_list": [{"type": "GenerateHybridOctreeHexahedraElementsWithCellColor",
                                  "model_part_name": "Carved", "color": 1}],
    "model_part_operations"  : []
}
"""))
mod_carved.SetupModelPart()
n_inside = model_carved.GetModelPart("Carved").NumberOfElements()

print(f"Total octree cells (no coloring) : {n_all}")
print(f"Inside cells (color=1)           : {n_inside}")
print(f"Outside cells removed            : {n_all - n_inside}")
print(f"Carve ratio (inside/total)       : {n_inside / n_all:.1%}")

# Side-by-side visualisation
grid_all    = pv_utils.ModelPartToPyVista(model_all.GetModelPart("All"))
grid_carved = pv_utils.ModelPartToPyVista(model_carved.GetModelPart("Carved"))

pl = pv.Plotter(shape=(1, 2))
pl.subplot(0, 0)
pl.add_mesh(grid_all,    color="steelblue",  show_edges=True)
pl.add_text("All cells (no coloring)", font_size=10)
pl.subplot(0, 1)
pl.add_mesh(grid_carved, color="tomato",     show_edges=True)
pl.add_text("Inside cells only",            font_size=10)
pl.link_views()
pl.show()

### Step 4: Boundary Conditions

`GenerateHybridOctreeQuadrilateralConditionsWithFaceColor` scans every hex cell coloured `1` and emits one
`SurfaceCondition3D4N` per quad face that is **owned by exactly one hex** — i.e. faces on the
outer boundary of the carved region.

The conditions are placed in a separate `ModelPart` (`"Boundary"`) so they can be assigned
boundary values independently from the volume mesh.  Every boundary node is also present in the
volume `ModelPart` (shared node IDs), so no duplication occurs.

In [ ]:
model_bc = KM.Model()
build_closed_box_surface(model_bc, name="Surface")

mod_bc = KM.OctreeHybridMeshGeneratorModeler(model_bc, KM.Parameters("""
{
    "input_model_part_name"  : "Surface",
    "refine_operations_list" : [
        { "type": "OctreeHybridRefineInterfaceCells", "adaptive": false },
        { "type": "OctreeHybridRefineUniform", "refinement_depth": 4 }
    ],
    "coloring_settings_list"  : [{"type": "OctreeHybridClassifyCellsInsideOutside"}],
    "entities_generator_list" : [
        {"type": "GenerateHybridOctreeHexahedraElementsWithCellColor",
         "model_part_name": "Volume",   "color": 1},
        {"type": "GenerateHybridOctreeQuadrilateralConditionsWithFaceColor",
         "model_part_name": "Boundary", "color": 1}
    ],
    "model_part_operations" : []
}
"""))
mod_bc.SetupModelPart()

mp_vol_bc = model_bc.GetModelPart("Volume")
mp_bnd    = model_bc.GetModelPart("Boundary")

print(f"Volume  : {mp_vol_bc.NumberOfElements()} hexes,  {mp_vol_bc.NumberOfNodes()} nodes")
print(f"Boundary: {mp_bnd.NumberOfConditions()} quads,  {mp_bnd.NumberOfNodes()} nodes")
print(f"Max possible boundary faces (6 × n_elem)  : {6 * mp_vol_bc.NumberOfElements()}")
print(f"Actual boundary faces                     : {mp_bnd.NumberOfConditions()}")
print(f"  (ratio < 1 confirms interior faces are shared)")

# Check: every boundary node id is also in the volume mesh
vol_ids = {n.Id for n in mp_vol_bc.Nodes}
bnd_ids = {n.Id for n in mp_bnd.Nodes}
assert bnd_ids.issubset(vol_ids), "Boundary nodes not a subset of volume nodes!"
print("Node-id consistency check: PASSED (boundary ⊆ volume)")

In [ ]:
# Visualise: volume wireframe + extracted outer surface coloured tomato
vol_grid_bc = pv_utils.ModelPartToPyVista(mp_vol_bc)
outer_surf  = pv_utils.CreateExtractedSurface(mp_vol_bc)

plotter_bc = pv.Plotter()
plotter_bc.add_mesh(vol_grid_bc, style="wireframe", color="steelblue",
                    opacity=0.35, label="Volume hexes")
plotter_bc.add_mesh(outer_surf,  color="tomato", show_edges=True,
                    opacity=0.85, label="Outer surface")
plotter_bc.add_legend()
plotter_bc.add_text("Volume mesh + outer boundary surface", font_size=11)
plotter_bc.show()

### Step 5: Adaptive Mesh and Quality Report

Setting `"adaptive": true` allows the octree engine to selectively refine cells near the input
surface, producing a graded mesh: fine near the surface, coarser in the interior.  This reduces
the element count compared to a uniform octree of the same maximum depth.

With `"tag_refinement_level": true`, every element stores its octree level in the non-historical
variable `REFINEMENT_LEVEL` (accessed via `element.GetValue(KM.REFINEMENT_LEVEL)`).  Template
hexes at refinement transitions receive the sentinel value `−1`.

The `OctreeHybridReportMeshQuality` operation logs minimum scaled-Jacobian statistics at the end of the pipeline.

In [ ]:
model_adapt = KM.Model()
build_closed_box_surface(model_adapt, name="Surface")

mod_adapt = KM.OctreeHybridMeshGeneratorModeler(model_adapt, KM.Parameters("""
{
    "input_model_part_name"  : "Surface",
    "refine_operations_list" : [
        {
            "type"            : "OctreeHybridRefineInterfaceCells",
            "refinement_depth": 5,
            "adaptive"        : true,
            "mesh_type"       : "dual"
        }
    ],
    "coloring_settings_list"  : [{"type": "OctreeHybridClassifyCellsInsideOutside"}],
    "entities_generator_list" : [{
        "type"                : "GenerateHybridOctreeHexahedraElementsWithCellColor",
        "model_part_name"     : "Volume",
        "color"               : 1,
        "tag_refinement_level": true
    }],
    "model_part_operations" : [
        {"type": "OctreeHybridReportMeshQuality", "model_part_name": "Volume"}
    ]
}
"""))
mod_adapt.SetupModelPart()

mp_adapt = model_adapt.GetModelPart("Volume")
print(f"Adaptive dual mesh (depth=5):")
print(f"  Nodes    : {mp_adapt.NumberOfNodes()}")
print(f"  Elements : {mp_adapt.NumberOfElements()}")

levels = [el.GetValue(KM.REFINEMENT_LEVEL) for el in mp_adapt.Elements]
level_dist = dict(sorted(collections.Counter(levels).items()))
print(f"  REFINEMENT_LEVEL distribution: {level_dist}")
print(f"    (−1 = transition-template hex, positive = leaf level)")

In [ ]:
# Visualise adaptive mesh coloured by REFINEMENT_LEVEL (cell data)
grid_adapt = pv_utils.ModelPartToPyVista(mp_adapt)
grid_adapt.cell_data["REFINEMENT_LEVEL"] = np.array(
    [el.GetValue(KM.REFINEMENT_LEVEL) for el in mp_adapt.Elements]
)

grid_adapt.plot(
    scalars="REFINEMENT_LEVEL",
    show_edges=True,
    cmap="tab10",
    cpos="iso",
    text="Adaptive Dual Mesh — coloured by octree refinement level",
)

### Step 6: Primal Mesh with Hanging-Node Constraints

Setting `"mesh_type": "primal"` produces one hexahedron per octree **leaf cell** (rather than per
primal vertex).  This is faster to generate and avoids the dual extraction + transition-template
pass, but the mesh is **non-conforming** at 2:1 refinement boundaries: finer cells introduce
*hanging nodes* that lie on the face or edge of a coarser neighbour without being one of its
corner nodes.

`GenerateHybridOctreeHexahedraElementsWithCellColor` resolves these by emitting a `LinearMasterSlaveConstraint` for
each hanging DOF:

$$u_s = \sum_{m} w_m \, u_m$$

where the bilinear interpolation weights $w_m$ satisfy the **partition-of-unity** property
$\sum_m w_m = 1$.  Hanging nodes on an edge have 2 masters; face-centre hanging nodes have 4.

In [ ]:
# Use the transition surface to guarantee 2:1 refinement transitions
model_primal = KM.Model()
build_transition_surface(model_primal, name="Surface")

mod_primal = KM.OctreeHybridMeshGeneratorModeler(model_primal, KM.Parameters("""
{
    "input_model_part_name"  : "Surface",
    "refine_operations_list" : [
        {
            "type"            : "OctreeHybridRefineInterfaceCells",
            "refinement_depth": 4,
            "adaptive"        : true,
            "mesh_type"       : "primal"
        }
    ],
    "coloring_settings_list"  : [],
    "entities_generator_list" : [
        {
            "type"               : "GenerateHybridOctreeHexahedraElementsWithCellColor",
            "model_part_name"    : "Volume",
            "color"              : 1,
            "tag_refinement_level": true,
            "variables"          : ["DISPLACEMENT_X", "DISPLACEMENT_Y", "DISPLACEMENT_Z"]
        }
    ],
    "model_part_operations" : []
}
"""))
mod_primal.SetupModelPart()

mp_primal = model_primal.GetModelPart("Volume")
nc = mp_primal.NumberOfMasterSlaveConstraints()
print(f"Primal mesh (adaptive, depth=4):")
print(f"  Nodes       : {mp_primal.NumberOfNodes()}")
print(f"  Elements    : {mp_primal.NumberOfElements()}")
print(f"  Constraints : {nc}  "
      f"({nc // 3} hanging nodes × 3 DOF variables)")

In [ ]:
# Verify partition-of-unity and master-count distribution
n_fail       = 0
master_counts = []

for c in mp_primal.MasterSlaveConstraints:
    T, b = KM.Matrix(), KM.Vector()
    c.CalculateLocalSystem(T, b, KM.ProcessInfo())
    row_sum = sum(T[0, j] for j in range(T.Size2()))
    master_counts.append(T.Size2())
    if abs(row_sum - 1.0) > 1e-10:
        n_fail += 1

mc_dist = dict(sorted(collections.Counter(master_counts).items()))
print(f"Partition-of-unity violations: {n_fail} / {nc}")
print(f"Master-count distribution    : {mc_dist}")
print(f"  2-master = edge-midpoint hanging node")
print(f"  4-master = face-centre  hanging node")

In [ ]:
# Visualise primal mesh coloured by REFINEMENT_LEVEL
grid_primal = pv_utils.ModelPartToPyVista(mp_primal)
grid_primal.cell_data["REFINEMENT_LEVEL"] = np.array(
    [el.GetValue(KM.REFINEMENT_LEVEL) for el in mp_primal.Elements]
)

grid_primal.plot(
    scalars="REFINEMENT_LEVEL",
    show_edges=True,
    cmap="tab10",
    cpos="iso",
    text="Primal Mesh — refinement level (non-conforming at transitions)",
)

### Step 7: Dual Mesh — Conformity Comparison

A useful sanity check is to run **both** the dual and primal topologies on the same surface
and compare element counts.  For the same `refinement_depth`:

- The **primal** mesh has exactly one hex per leaf cell (leaf count = element count).
- The **dual** mesh has one hex per interior primal vertex — fewer cells in a uniform octree,
  but additional transition-template hexes near boundaries mean the counts can differ.

The dual mesh never creates hanging-node constraints; the primal mesh always does when
`adaptive=true`.

In [ ]:
def run_and_count(mesh_type, adaptive, depth=4):
    """Build a transition-surface mesh and return element/constraint counts."""
    m = KM.Model()
    build_transition_surface(m, name="S")
    coloring = '[{"type":"OctreeHybridClassifyCellsInsideOutside"}]' if mesh_type == "dual" else '[]'
    if mesh_type == "primal":
        gen_json = '[{"type":"GenerateHybridOctreeHexahedraElementsWithCellColor","model_part_name":"O","color":1,' \
                   '"variables":["DISPLACEMENT_X"]}]'
    else:
        gen_json = '[{"type":"GenerateHybridOctreeHexahedraElementsWithCellColor","model_part_name":"O","color":1}]'  
    adaptive_str = "true" if adaptive else "false"
    settings = KM.Parameters(f"""{{
        "input_model_part_name":"S","refine_operations_list":[{{"type":"OctreeHybridRefineInterfaceCells",
                                    "refinement_depth":{depth},
                                    "adaptive":{adaptive_str},"mesh_type":"{mesh_type}"}}],
        "coloring_settings_list":{coloring},
        "entities_generator_list":{gen_json},
        "model_part_operations":[]
    }}""")
    KM.OctreeHybridMeshGeneratorModeler(m, settings).SetupModelPart()
    out = m.GetModelPart("O")
    return out.NumberOfElements(), out.NumberOfMasterSlaveConstraints()


for depth in (3, 4, 5):
    n_dual,   nc_dual   = run_and_count("dual",   adaptive=True,  depth=depth)
    n_primal, nc_primal = run_and_count("primal", adaptive=True,  depth=depth)
    print(f"depth={depth}  |  dual: {n_dual:5d} elems, {nc_dual:4d} constraints  "
          f"|  primal: {n_primal:5d} elems, {nc_primal:4d} constraints")

### Step 8: Surface-Conforming Mesh via Jacobian-Controlled Projection

Setting `"project_to_surface": true` in the `OctreeHybridRefineInterfaceCells` entry activates
three additional pipeline passes **inside the octree generator** (before the entity-generation
stage):

1. **`RemoveOutsideElement`** — signed-distance ray-cast carve: retains only hexes
   whose nodes are predominantly inside the input surface.  Replaces the
   `OctreeHybridClassifyCellsInsideOutside` coloring stage (which is therefore
   **not needed** here).
2. **`ClearBufferZone`** — hemisphere test that removes folded boundary hexes
   (avoids locked inverted slivers during optimisation).
3. **`ProjectToIsoSurface`** — Jacobian-controlled gradient-descent + Laplacian
   smoothing that displaces boundary nodes onto the input surface while maximising
   scaled-Jacobian quality.  Budget is controlled by:
   - `"projection_iterations"` (default `20000`) — gradient-descent steps;
     more iterations → higher worst-element quality (monotone convergence).
   - `"projection_smoothing"` (default `1000`) — Laplacian smoothing frequency.

The buffer-layer hexes (level `−2` in `REFINEMENT_LEVEL`) are the thin shell added
between the carved interior and the target surface.  Core hexes keep their original
level values.  The result is a **surface-conforming** mesh with no hanging nodes and
0 inverted core elements.

> **Only `mesh_type: "dual"` supports `project_to_surface`.**

In [ ]:
# Use the bunny if available, otherwise fall back to the unit box
bunny_path_proj = os.path.join(
    notebook_dir, "..", "..", "tests",
    "auxiliar_files_for_python_unittest", "stl_files", "Bunny-LowPoly.stl"
)

model_proj = KM.Model()
if os.path.exists(bunny_path_proj):
    mp_surf = model_proj.CreateModelPart("Surface")
    mp_surf.ProcessInfo[KM.DOMAIN_SIZE] = 3
    KM.StlIO(bunny_path_proj, KM.Parameters('{"open_mode":"read"}')).ReadModelPart(mp_surf)
    surface_name = "Surface"
    output_name  = "BunnyVolume"
    depth        = 6
    print(f"Using Bunny-LowPoly.stl: {mp_surf.NumberOfNodes()} nodes, "
          f"{mp_surf.NumberOfGeometries()} triangles")
else:
    build_closed_box_surface(model_proj, name="Surface")
    surface_name = "Surface"
    output_name  = "Volume"
    depth        = 5
    print("Bunny STL not found — using unit box fallback")

mod_proj = KM.OctreeHybridMeshGeneratorModeler(model_proj, KM.Parameters(f"""
{{
    \"input_model_part_name\"  : \"{surface_name}\",
    \"refine_operations_list\" : [{{
        \"type\"                 : \"OctreeHybridRefineInterfaceCells\",
        \"refinement_depth\"     : {depth},
        \"adaptive\"             : true,
        \"mesh_type\"            : \"dual\",
        \"project_to_surface\"   : true,
        \"projection_iterations\": 20000,
        \"projection_smoothing\" : 1000
    }}],
    \"coloring_settings_list\"  : [],
    \"entities_generator_list\" : [{{
        \"type\"                : \"GenerateHybridOctreeHexahedraElementsWithCellColor\",
        \"model_part_name\"     : \"{output_name}\",
        \"color\"               : 1,
        \"tag_refinement_level\": true
    }}],
    \"model_part_operations\" : [
        {{\"type\": \"OctreeHybridReportMeshQuality\", \"model_part_name\": \"{output_name}\"}}
    ]
}}
"""))
mod_proj.SetupModelPart()

mp_proj = model_proj.GetModelPart(output_name)
print(f"Projected dual mesh (depth={depth}):")
print(f"  Nodes    : {mp_proj.NumberOfNodes()}")
print(f"  Elements : {mp_proj.NumberOfElements()}")

levels_proj = [el.GetValue(KM.REFINEMENT_LEVEL) for el in mp_proj.Elements]
level_dist_proj = dict(sorted(collections.Counter(levels_proj).items()))
print(f"  REFINEMENT_LEVEL distribution: {level_dist_proj}")
print(f"    (−2 = buffer-zone shell, −1 = transition template, positive = leaf level)")

grid_proj = pv_utils.ModelPartToPyVista(mp_proj)
grid_proj.cell_data["REFINEMENT_LEVEL"] = np.array(
    [el.GetValue(KM.REFINEMENT_LEVEL) for el in mp_proj.Elements]
)
grid_proj.plot(
    scalars="REFINEMENT_LEVEL",
    show_edges=True,
    cmap="tab10",
    cpos="iso",
    text=f"Projected Dual Mesh ({output_name}) — surface-conforming, depth={depth}",
)

### Step 9: Loading a Surface from STL (Optional)

Any closed, orientable surface can be loaded from an STL file using `KM.StlIO`.  The cell below
looks for `Bunny-LowPoly.stl` in the Kratos test directory; if found it runs the full dual-mesh
pipeline and reports statistics.  The test is skipped gracefully when the file is absent.

Both ASCII and binary STL files are accepted — the helper converts binary files on the fly.

In [ ]:
def load_stl_surface(model, stl_path, name="Surface"):
    """Load a surface mesh from an STL file (ASCII or binary) into a new ModelPart."""
    mp = model.CreateModelPart(name)
    mp.ProcessInfo[KM.DOMAIN_SIZE] = 3
    KM.StlIO(stl_path, KM.Parameters('{"open_mode":"read"}')).ReadModelPart(mp)
    return mp


bunny_path = os.path.join(
    notebook_dir, "..", "..", "tests",
    "auxiliar_files_for_python_unittest", "stl_files", "Bunny-LowPoly.stl"
)

if os.path.exists(bunny_path):
    model_stl = KM.Model()
    mp_stl = load_stl_surface(model_stl, bunny_path, name="Bunny")
    print(f"Loaded STL: {mp_stl.NumberOfNodes()} nodes, "
          f"{mp_stl.NumberOfGeometries()} triangles")

    mod_stl = KM.OctreeHybridMeshGeneratorModeler(model_stl, KM.Parameters("""
    {
        "input_model_part_name"  : "Bunny",
        "refine_operations_list" : [
            {
                "type"      : "OctreeHybridRefineInterfaceCells",
                "adaptive"  : false,
                "mesh_type" : "dual"
            },
            { "type"                  : "OctreeHybridRefineInterfaceCells",
              "input_model_part_name" : "Bunny",
              "refinement_depth"      : 4 }
        ],
        "coloring_settings_list"  : [{"type": "OctreeHybridClassifyCellsInsideOutside"}],
        "entities_generator_list" : [{
            "type"                : "GenerateHybridOctreeHexahedraElementsWithCellColor",
            "model_part_name"     : "BunnyVolume",
            "color"               : 1,
            "tag_refinement_level": true
        }],
        "model_part_operations" : [
            {"type": "OctreeHybridReportMeshQuality", "model_part_name": "BunnyVolume"}
        ]
    }
    """))
    mod_stl.SetupModelPart()

    mp_bunny = model_stl.GetModelPart("BunnyVolume")
    print(f"Bunny hex mesh: {mp_bunny.NumberOfElements()} elements, "
          f"{mp_bunny.NumberOfNodes()} nodes")

    grid_bunny = pv_utils.ModelPartToPyVista(mp_bunny)
    grid_bunny.cell_data["REFINEMENT_LEVEL"] = np.array(
        [el.GetValue(KM.REFINEMENT_LEVEL) for el in mp_bunny.Elements]
    )
    grid_bunny.plot(
        scalars="REFINEMENT_LEVEL",
        cmap="tab10",
        show_edges=True,
        cpos="iso",
        text="Bunny Hex Mesh — interface-cell refinement, depth=4",
    )
else:
    print(f"Bunny-LowPoly.stl not found at:\n  {os.path.normpath(bunny_path)}")
    print("Provide any closed STL surface and update 'bunny_path' to run this cell.")

### Step 10: Exporting to VTU / VTK

Two export paths are available:

1. **PyVista `.save()`** — writes the `pyvista.UnstructuredGrid` directly to a `.vtu` file,
   including any point- or cell-data arrays that were added (e.g. `REFINEMENT_LEVEL`).
2. **`pv_utils.SaveModelPart`** — Kratos-native helper that converts the `ModelPart` to a grid
   and writes it, preserving any nodal `Variable` arrays listed in `nodalVariables`.

Both formats can be opened directly in **ParaView**.

In [ ]:
temp_dir = tempfile.gettempdir()

# --- Option 1: save the PyVista grid (includes REFINEMENT_LEVEL cell data) ---
vtu_pv = os.path.join(temp_dir, "hex_mesh_adaptive.vtu")
grid_adapt.save(vtu_pv)
print(f"PyVista grid saved to : {vtu_pv}")

# --- Option 2: Kratos SaveModelPart (mesh geometry only, no nodal variables here) ---
vtu_km = os.path.join(temp_dir, "hex_mesh_modelpart.vtu")
pv_utils.SaveModelPart(mp_adapt, vtu_km)
print(f"Kratos ModelPart saved: {vtu_km}")

# --- Option 3: low-level OctreeHybridMeshUtility VTK writer (legacy format) ---
vtk_util = os.path.join(temp_dir, "hex_mesh_utility.vtk")
model_vtk = KM.Model()
build_closed_box_surface(model_vtk, name="Surface")
KM.OctreeHybridMeshUtility.BuildAndWriteVtk(
    model_vtk.GetModelPart("Surface"), vtk_util, 4
)
print(f"Utility VTK saved to  : {vtk_util}")
print("\nAll three files can be opened in ParaView.")
print("Colour by 'REFINEMENT_LEVEL' (option 1) or 'level' (option 3) to see adaptive refinement.")

### Step 11: Tetrahedral Mesh (BCC Freudenthal Decomposition)

Each hex cell in the dual mesh is decomposed into **6 tetrahedra** using the
Freudenthal–Kuhn scheme along the main diagonal (local nodes 0 → 6).  This implements
the BCC-lattice tetrahedral pattern from §3.3.3 of the thesis: the dual-hex cell centres
act as BCC body positions, yielding tetrahedra with a minimum dihedral angle of **45°**.

The six tets sharing the (0, 6) diagonal are:

| Tet | Nodes (local hex indices) |
|-----|--------------------------|
| 0   | 0, 3, 6, 2               |
| 1   | 3, 6, 7, 0               |
| 2   | 4, 7, 6, 0               |
| 3   | 0, 4, 5, 6               |
| 4   | 0, 1, 2, 6               |
| 5   | 1, 5, 6, 0               |

Boundary conditions are **triangles** obtained by splitting each outer quad `{n₀, n₁, n₂, n₃}`
along the `(n₀, n₂)` diagonal — consistent with the tet decomposition so the combined mesh is
**conforming**: every boundary triangle is an exposed face of a boundary tetrahedron.

Two new entity-generation stages are used:
- `GenerateHybridOctreeTetrahedraElementsWithCellColor` — emits `Element3D4N` (6 per hex).
- `GenerateHybridOctreeTriangularConditionsWithFaceColor` — emits `SurfaceCondition3D3N` (2 per boundary quad).

In [ ]:
# --- Step 11: Tetrahedral mesh + triangular boundary conditions ---

model_tet = KM.Model()
build_closed_box_surface(model_tet, name="SkinTet")

settings_tet = KM.Parameters("""
{
    "input_model_part_name"  : "SkinTet",
    "refine_operations_list" : [{
        "type"            : "OctreeHybridRefineInterfaceCells",
        "refinement_depth": 4,
        "adaptive"        : false
    }],
    "coloring_settings_list" : [{ "type": "OctreeHybridClassifyCellsInsideOutside" }],
    "entities_generator_list": [
        {
            "type"                 : "GenerateHybridOctreeTetrahedraElementsWithCellColor",
            "model_part_name"      : "TetMesh",
            "color"                : 1,
            "tag_refinement_level" : true
        },
        {
            "type"             : "GenerateHybridOctreeTriangularConditionsWithFaceColor",
            "model_part_name"  : "TetMesh.Boundary",
            "color"            : 1
        }
    ],
    "model_part_operations" : []
}
""")

modeler_tet = KM.OctreeHybridMeshGeneratorModeler(model_tet, settings_tet)
modeler_tet.SetupModelPart()

tet_mp   = model_tet.GetModelPart("TetMesh")
bound_mp = model_tet.GetModelPart("TetMesh.Boundary")

print(f"Tet mesh:")
print(f"  Elements : {tet_mp.NumberOfElements()}")
print(f"  Nodes    : {tet_mp.NumberOfNodes()}")
print(f"Boundary:")
print(f"  Conditions: {bound_mp.NumberOfConditions()}")
print(f"  Nodes     : {bound_mp.NumberOfNodes()}")

# Cross-check: tet count must be exactly 6 × hex count for the same settings
model_hex_ref = KM.Model()
build_closed_box_surface(model_hex_ref, name="SkinHex")
settings_hex_ref = KM.Parameters("""
{
    "input_model_part_name"  : "SkinHex",
    "refine_operations_list" : [{
        "type"            : "OctreeHybridRefineInterfaceCells",
        "refinement_depth": 4,
        "adaptive"        : false
    }],
    "coloring_settings_list" : [{ "type": "OctreeHybridClassifyCellsInsideOutside" }],
    "entities_generator_list": [{
        "type"            : "GenerateHybridOctreeHexahedraElementsWithCellColor",
        "model_part_name" : "HexMesh",
        "color"           : 1
    }],
    "model_part_operations" : []
}
""")
KM.OctreeHybridMeshGeneratorModeler(model_hex_ref, settings_hex_ref).SetupModelPart()
n_hexes = model_hex_ref.GetModelPart("HexMesh").NumberOfElements()

print(f"\nHex reference: {n_hexes} hexes")
print(f"Tet / Hex ratio: {tet_mp.NumberOfElements()} / {n_hexes} = "
      f"{tet_mp.NumberOfElements() / n_hexes:.1f}×  (expected: 6.0×)")
print(f"Tri / Hex-face ratio: {bound_mp.NumberOfConditions()} / "
      f"{model_hex_ref.GetModelPart('HexMesh').NumberOfConditions() if False else '—'}")

# Verify: no inverted tetrahedra (positive signed volume for all)
n_inv = 0
for el in tet_mp.Elements:
    g = el.GetGeometry()
    e0 = [g[1].X - g[0].X, g[1].Y - g[0].Y, g[1].Z - g[0].Z]
    e1 = [g[2].X - g[0].X, g[2].Y - g[0].Y, g[2].Z - g[0].Z]
    e2 = [g[3].X - g[0].X, g[3].Y - g[0].Y, g[3].Z - g[0].Z]
    vol = (e0[0] * (e1[1]*e2[2] - e1[2]*e2[1])
         - e0[1] * (e1[0]*e2[2] - e1[2]*e2[0])
         + e0[2] * (e1[0]*e2[1] - e1[1]*e2[0]))
    if vol <= 0.0:
        n_inv += 1
print(f"\nInverted tetrahedra: {n_inv}  (expected: 0)")

In [ ]:
# Visualise: tet mesh wireframe + triangle boundary conditions
try:
    import pyvista as pv

    pts_tet = [(n.X, n.Y, n.Z) for n in tet_mp.Nodes]
    node_map_tet = {n.Id: i for i, n in enumerate(tet_mp.Nodes)}
    cells_tet = []
    for el in tet_mp.Elements:
        cells_tet += [4] + [node_map_tet[n.Id] for n in el.GetNodes()]

    pts_bc = [(n.X, n.Y, n.Z) for n in bound_mp.Nodes]
    node_map_bc = {n.Id: i for i, n in enumerate(bound_mp.Nodes)}
    cells_bc = []
    for cond in bound_mp.Conditions:
        cells_bc += [3] + [node_map_bc[n.Id] for n in cond.GetNodes()]

    mesh_tet = pv.UnstructuredGrid(
        cells_tet,
        [pv.CellType.TETRA] * tet_mp.NumberOfElements(),
        pts_tet,
    )
    mesh_bc = pv.UnstructuredGrid(
        cells_bc,
        [pv.CellType.TRIANGLE] * bound_mp.NumberOfConditions(),
        pts_bc,
    )

    # Add REFINEMENT_LEVEL as cell data on the tet mesh
    mesh_tet.cell_data["REFINEMENT_LEVEL"] = np.array(
        [el.GetValue(KM.REFINEMENT_LEVEL) for el in tet_mp.Elements]
    )

    pl = pv.Plotter()
    pl.add_mesh(
        mesh_tet,
        scalars="REFINEMENT_LEVEL",
        cmap="tab10",
        show_edges=True,
        opacity=0.6,
        label="Tet mesh (coloured by refinement level)",
    )
    pl.add_mesh(
        mesh_bc,
        color="tomato",
        opacity=0.85,
        show_edges=True,
        label="Triangle boundary conditions",
    )
    pl.add_legend()
    pl.add_text(
        "BCC Freudenthal Tet Mesh — 6 tets per hex, triangular BCs on boundary",
        font_size=10,
    )
    pl.show()

except ImportError:
    print("PyVista not available — skipping visualisation.")
    print("Install with: pip install pyvista")